PRIMEIRO IREMOS CARREGAR AS TABELAS QUE ESTÃO NA SILVER PARA TRABALHAR COM ELAS


1.1 Criação da tabela gold.ft_vendas_consumidor_local

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.ft_vendas_consumidor_local AS 
SELECT 
  p.id_pedido, 
  p.id_consumidor, 
  CAST(p.valor_total_pago_brl AS DECIMAL(12,2)) AS valor_total_pago_brl, 
  CAST(p.data_pedido AS DATE) AS data_pedido, 
  c.cidade, 
  c.estado 
  FROM ecommerce.silver.ft_pedido_total p 
  LEFT JOIN ecommerce.silver.ft_consumidores c 
    ON p.id_consumidor = c.id_consumidor;

1.2 Criação da view
gold.view_total_compras_por_consumidor

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_total_compras_por_consumidor AS 
SELECT 
  cidade, 
  estado, 
  COUNT(*) AS quantidade_vendas, 
  SUM(valor_total_pago_brl) AS valor_total_localidade
FROM ecommerce.gold.ft_vendas_consumidor_local 
GROUP BY cidade, estado;


Respondendo perguntas da área de negócio:
Crie uma consulta SQL para exibir o total de vendas por estado

In [0]:
%sql
SELECT
    estado,
    SUM(valor_total_localidade) AS `total vendas por estado`
FROM ecommerce.gold.view_total_compras_por_consumidor
GROUP BY estado
ORDER BY `total vendas por estado` DESC;


2º Projeto — Área de Logística (Análise de Atrasos
de Entregas)

## 2.1 Criação da tabela
## gold.ft_atrasos_pedidos_local_vendedor:
## Cada linha representa um pedido com suas informações logísticas
### básicas.
### As informações devem ser obtidas a partir de:
### silver.ft_pedidos
### silver.ft_consumidores
### silver.ft_itens_pedidos

In [0]:
%sql
DESCRIBE ecommerce.silver.ft_pedidos;


In [0]:
%sql
DESCRIBE ecommerce.silver.ft_consumidores;


In [0]:
%sql
DESCRIBE ecommerce.silver.ft_itens_pedidos;


PRIMEIRO IREI DAR UM JOIN NA TABELA DE CONSUMIDORES, COM A DE PEDIDOS PARA DAR OUTRO JOIN NESSE RESULTADO

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.ft_atrasos_pedidos_local_vendedor
SELECT
  p.id_pedido,
  i.id_vendedor,
  p.id_consumidor,
  p.entregue_no_prazo,
  p.tempo_entrega_dias,
  p.tempo_entrega_estimado_dias,
  c.cidade,
  c.estado
  FROM ecommerce.silver.ft_pedidos p
  LEFT JOIN ecommerce.silver.ft_consumidores c
    ON p.id_consumidor = c.id_consumidor
  LEFT JOIN ecommerce.silver.ft_itens_pedidos i
    ON p.id_pedido = i.id_pedido


2.2 Criação das Views Analíticas

2.2.1 gold.view_tempo_medio_entrega_localidade

irei usar o spark para construir a view em si

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_tempo_medio_entrega_localidade AS
SELECT
    cidade,
    estado,
    ROUND(AVG(tempo_entrega_dias), 2) AS tempo_medio_entrega,
    ROUND(AVG(tempo_entrega_estimado_dias), 2) AS tempo_medio_estimado,
    CASE 
        WHEN MAX(
            CASE 
                WHEN tempo_entrega_dias > tempo_entrega_estimado_dias THEN 1
                ELSE 0
            END
        ) = 1 THEN 'SIM'
        ELSE 'NÃO'
    END AS entrega_maior_que_estimado
FROM ecommerce.gold.ft_atrasos_pedidos_local_vendedor
GROUP BY cidade, estado;


2.2.2 gold.view_vendedor_pontualidade


In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_vendedor_pontualidade AS
SELECT
    id_vendedor,
    COUNT(*) as total_pedidos,
    COUNT(CASE WHEN entregue_no_prazo = 'NÃO' THEN 1 END) as pedidos_entregues_no_prazo,
    ROUND(
    COUNT(CASE WHEN tempo_entrega_dias > tempo_entrega_estimado_dias THEN 1 END) 
    / COUNT(*) * 100
, 2) AS percentual_atraso
FROM ecommerce.gold.ft_atrasos_pedidos_local_vendedor
GROUP BY id_vendedor

3º Projeto — Área Comercial (Análises de Vendas
por Período)

3.1 Criação da Dimensão de Tempo — gold.dm_tempo
Será necessário criar uma dimensão, para que auxilie nas análises temporais
em diferentes granularidades (ano, trimestre, mês, semana, dia, etc.). Utilize as
funções explode e sequence para gerar os valores entre as datas de início e fim.


PRIMEIRO IREI DESCOBRIR O RANGE DOS MEUS PEDIDOS

In [0]:


%sql
SELECT 
    MIN(data_pedido) AS data_inicio,
    MAX(data_pedido) AS data_fim
FROM ecommerce.gold.ft_vendas_consumidor_local;


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.dm_tempo AS
SELECT
    dt AS sk_tempo,
    year(dt) AS ano,
    quarter(dt) AS trimestre,
    month(dt) AS mes,
    weekofyear(dt) AS semana_do_ano,
    day(dt) AS dia,
    
    dayofweek(dt) AS dia_da_semana_num,
    
    CASE dayofweek(dt)
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Segunda'
        WHEN 3 THEN 'Terça'
        WHEN 4 THEN 'Quarta'
        WHEN 5 THEN 'Quinta'
        WHEN 6 THEN 'Sexta'
        WHEN 7 THEN 'Sábado'
    END AS dia_da_semana_nome,

    CASE 
        WHEN dayofweek(dt) IN (1, 7) THEN 'Sim'
        ELSE 'Não'
    END AS eh_fim_de_semana,

    CASE month(dt)
        WHEN 1 THEN 'Janeiro'
        WHEN 2 THEN 'Fevereiro'
        WHEN 3 THEN 'Março'
        WHEN 4 THEN 'Abril'
        WHEN 5 THEN 'Maio'
        WHEN 6 THEN 'Junho'
        WHEN 7 THEN 'Julho'
        WHEN 8 THEN 'Agosto'
        WHEN 9 THEN 'Setembro'
        WHEN 10 THEN 'Outubro'
        WHEN 11 THEN 'Novembro'
        WHEN 12 THEN 'Dezembro'
    END AS mes_nome

FROM (
    SELECT explode(
        sequence(
            to_date('2016-09-04'),
            to_date('2018-10-17'),
            interval 1 day
        )
    ) AS dt
);


3.2 Criação da Fato gold.ft_vendas_geral


In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.ft_vendas_geral AS
SELECT

    p.id_pedido, 
    it.id_item, 

    c.id_consumidor      AS fk_cliente, 
    pr.id_produto        AS fk_produto, 
    v.id_vendedor        AS fk_vendedor, 
    DATE(p.pedido_compra_timestamp)  AS fk_tempo, 

    p.status             AS status_pedido,
    p.tempo_entrega_dias, 
    p.entregue_no_prazo AS entrega_no_prazo, 

    it.preco_BRL   AS valor_produto_brl, 
    it.preco_frete AS valor_frete_brl, 
    (it.preco_BRL + it.preco_frete) AS valor_total_item_brl, 

    ROUND(it.preco_BRL   / cot.cotacao_dolar, 2) AS valor_produto_usd,
    ROUND(it.preco_frete / cot.cotacao_dolar, 2) AS valor_frete_usd,
    ROUND((it.preco_BRL + it.preco_frete) / cot.cotacao_dolar, 2) AS valor_total_item_usd,
    cot.cotacao_dolar, 
    av.avaliacao AS avaliacao_pedido

FROM ecommerce.silver.ft_pedidos p

INNER JOIN ecommerce.silver.ft_itens_pedidos it
    ON p.id_pedido = it.id_pedido

LEFT JOIN ecommerce.silver.ft_consumidores c
    ON p.id_consumidor = c.id_consumidor

LEFT JOIN ecommerce.silver.ft_produtos pr
    ON it.id_produto = pr.id_produto

LEFT JOIN ecommerce.silver.ft_vendedores v
    ON it.id_vendedor = v.id_vendedor

LEFT JOIN ecommerce.silver.dm_cotacao_dolar cot
    ON DATE(p.pedido_compra_timestamp) = cot.data 

LEFT JOIN ecommerce.silver.ft_avaliacoes_pedidos av
    ON p.id_pedido = av.id_pedido;


In [0]:
%sql
SHOW TABLES IN ecommerce.silver;

3.3 Criação da view gold.view_vendas_por_periodo

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_vendas_por_periodo AS
SELECT
    t.ano,
    t.trimestre,
    t.mes,
    t.mes_nome,
    t.dia,
    t.dia_da_semana_num,

    COUNT(DISTINCT f.id_pedido) AS total_pedidos,

    COUNT(f.id_item) AS total_itens,

    SUM(f.valor_total_item_brl) AS receita_total_brl,

    SUM(f.valor_total_item_usd) AS receita_total_usd,

    ROUND(
        SUM(f.valor_total_item_brl) / COUNT(f.id_item),
        2
    ) AS ticket_medio_brl,

    ROUND(
        AVG(f.avaliacao_pedido),
        2
    ) AS avaliacao_media

FROM ecommerce.gold.ft_vendas_geral f

INNER JOIN ecommerce.gold.dm_tempo t
    ON f.fk_tempo = t.sk_tempo

GROUP BY
    t.ano,
    t.trimestre,
    t.mes,
    t.mes_nome,
    t.dia,
    t.dia_da_semana_num

ORDER BY
    t.ano,
    t.mes,
    t.dia;


3.3.1 Consultas Analíticas

1. Qual é o dia da semana com maior receita total em reais ( receita_total_brl )?

In [0]:
%sql
SELECT dia_da_semana_num, 
SUM(receita_total_brl) AS receita_total_brl
FROM ecommerce.gold.view_vendas_por_periodo
GROUP BY dia_da_semana_num
ORDER BY receita_total_brl DESC
LIMIT 1

2. Considerando o último ano disponível na dimensão de tempo, qual foi o
mês com maior ticket médio ( ticket_medio_brl )?


In [0]:
%sql
SELECT 
  ROUND(AVG(ticket_medio_brl), 2) AS ticket_medio,
  mes,
  mes_nome
  FROM ecommerce.gold.view_vendas_por_periodo
  WHERE ano = (
    SELECT MAX(ano)
    FROM ecommerce.gold.view_vendas_por_periodo
  )
  GROUP BY mes, mes_nome
  ORDER BY ticket_medio DESC
  LIMIT 1;

3.4. Criação da gold.view_top_produto

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_top_produto AS
SELECT
    CAST(f.fk_produto AS STRING) AS id_produto,
    CAST(p.categoria_produto AS STRING) AS categoria_produto,
    CAST(COUNT(f.id_item) AS BIGINT) AS quantidade_vendida,
    CAST(COUNT(DISTINCT f.id_pedido) AS BIGINT) AS total_pedidos,
    CAST(SUM(f.valor_total_item_brl) AS DECIMAL(12,2)) AS receita_brl,
    CAST(SUM(f.valor_total_item_usd) AS DECIMAL(12,2)) AS receita_usd,
    CAST(ROUND(AVG(f.valor_produto_brl), 2) AS DECIMAL(12,2)) AS preco_medio_brl,
    CAST(ROUND(AVG(f.avaliacao_pedido), 2) AS DECIMAL(3,2)) AS avaliacao_media,
    CAST(ROUND(AVG(p.peso_produto_gramas), 2) AS DECIMAL(8,2)) AS peso_medio_gramas

FROM ecommerce.gold.ft_vendas_geral f

LEFT JOIN ecommerce.silver.ft_produtos p
    ON f.fk_produto = p.id_produto

GROUP BY 
    f.fk_produto,
    p.categoria_produto
ORDER BY receita_brl DESC;


3.5 Criação da view_vendas_produtos_esteticos

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.view_vendas_produtos_esteticos AS
WITH base_fashion AS (
    SELECT
        f.*,
        p.categoria_produto,
        t.ano,
        t.mes
    FROM ecommerce.gold.ft_vendas_geral f
    LEFT JOIN ecommerce.silver.ft_produtos p
        ON f.fk_produto = p.id_produto
    INNER JOIN ecommerce.gold.dm_tempo t
        ON f.fk_tempo = t.sk_tempo
    WHERE p.categoria_produto LIKE 'fashion%'
)

SELECT
    CAST(ano AS INT) AS ano,
    CAST(mes AS INT) AS mes,
    CAST(categoria_produto AS STRING) AS categoria_produto,
    CAST(COUNT(DISTINCT id_pedido) AS BIGINT) AS total_pedidos,
    CAST(COUNT(id_item) AS BIGINT) AS total_itens_vendidos,
    CAST(SUM(valor_total_item_brl) AS DECIMAL(12,2)) AS receita_total_brl,
    CAST(SUM(valor_total_item_usd) AS DECIMAL(12,2)) AS receita_total_usd,
    CAST(ROUND(AVG(valor_produto_brl), 2) AS DECIMAL(12,2)) AS ticket_medio_brl,
    CAST(ROUND(AVG(valor_produto_usd), 2) AS DECIMAL(12,2)) AS ticket_medio_usd,
    CAST(ROUND(AVG(avaliacao_pedido), 2) AS DECIMAL(3,2)) AS avaliacao_media

FROM base_fashion
GROUP BY ano, mes, categoria_produto
ORDER BY ano, mes, categoria_produto;
